# QG-ORS trên Kaggle — repo ZIP và CIFAR-100 từ Input

Notebook này **không clone GitHub** và **không tự tải dataset**. Trước khi chạy:

1. Trong **Settings → Accelerator**, chọn GPU.
2. Tạo/đính kèm một Kaggle Dataset riêng tư chứa ZIP của repo QG-RSA. Kaggle có thể tự giải nén ZIP; notebook hỗ trợ cả ZIP còn nguyên lẫn thư mục đã giải nén.
3. Đính kèm một Input chứa thư mục CIFAR-100 chuẩn `cifar-100-python`.
4. Bật Internet nếu cần cài PennyLane hoặc tải pretrained weights của `timm`. Dataset CIFAR-100 vẫn chỉ được đọc từ Input.

> Khuyến nghị: chạy smoke trước. Với full run, đặt `TASKS_PER_SESSION = 1` hoặc `2`, lưu output, rồi gắn output của phiên trước làm Input để resume.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, zipfile

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/qgors_workspace')
OUTPUT_ROOT = Path('/kaggle/working/QGORS_runs')

# Chỉ cần điền khi auto-detect tìm thấy nhiều lựa chọn. Có thể là ZIP hoặc thư mục repo.
REPO_INPUT_HINT = None
# Trỏ trực tiếp tới thư mục `cifar-100-python` nếu có nhiều bản dataset.
CIFAR_INPUT_HINT = None
# Tuỳ chọn: thư mục QGORS_runs từ output của phiên Kaggle trước.
PREVIOUS_RUN_INPUT_HINT = None

SEED = 1993
TASKS_PER_SESSION = 1
RUN_SMOKE = True
RUN_FULL = False
FULL_VARIANT = 'qgors'  # qgors | classical

print('Các Input đang được gắn:')
for path in sorted(INPUT_ROOT.iterdir()):
    print(' -', path)

## 1. Tìm và giải nén repo

Cell này kiểm tra chống path traversal trước khi giải nén. Mọi file cần ghi đều nằm trong `/kaggle/working`.

In [ ]:
def is_project_root(path):
    path = Path(path)
    return (path / 'main.py').is_file() and (path / 'models').is_dir() and (path / 'exps').is_dir()

def zip_looks_like_repo(path):
    try:
        with zipfile.ZipFile(path) as archive:
            names = [name.replace('\\', '/') for name in archive.namelist()]
        roots = {name[:-len('main.py')].rstrip('/') for name in names if name.endswith('main.py')}
        return any(
            any(name.startswith(root + '/models/') for name in names)
            and any(name.startswith(root + '/exps/') for name in names)
            for root in roots
        )
    except zipfile.BadZipFile:
        return False

def safe_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            try:
                target.relative_to(destination)
            except ValueError as exc:
                raise ValueError(f'ZIP chứa đường dẫn không an toàn: {member.filename}') from exc
        archive.extractall(destination)

def discover_repo():
    if REPO_INPUT_HINT:
        candidates = [Path(REPO_INPUT_HINT)]
    else:
        zip_candidates = [p for p in INPUT_ROOT.rglob('*.zip') if zip_looks_like_repo(p)]
        dir_candidates = [p.parent for p in INPUT_ROOT.rglob('main.py') if is_project_root(p.parent)]
        candidates = sorted(set(zip_candidates + dir_candidates))
    if len(candidates) != 1:
        raise RuntimeError(f'Cần đúng một repo QG-RSA, tìm thấy {len(candidates)}: {candidates}. Hãy đặt REPO_INPUT_HINT.')
    return candidates[0]

repo_source = discover_repo()
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)

if repo_source.is_file():
    safe_extract_zip(repo_source, WORK_ROOT / 'repo')
    roots = [p.parent for p in (WORK_ROOT / 'repo').rglob('main.py') if is_project_root(p.parent)]
    if len(roots) != 1:
        raise RuntimeError(f'ZIP phải chứa đúng một project, tìm thấy: {roots}')
    PROJECT_DIR = roots[0]
else:
    PROJECT_DIR = WORK_ROOT / 'repo'
    shutil.copytree(repo_source, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('Repo input :', repo_source)
print('Project dir:', PROJECT_DIR)

## 2. Gắn CIFAR-100 từ `/kaggle/input`

Code gốc gọi `torchvision.datasets.CIFAR100('./data/datasets', download=True)`. Notebook tạo symlink đúng cấu trúc để integrity check thành công và không phát sinh download.

In [ ]:
def discover_cifar_directory():
    if CIFAR_INPUT_HINT:
        candidates = [Path(CIFAR_INPUT_HINT)]
    else:
        candidates = [p for p in INPUT_ROOT.rglob('cifar-100-python') if p.is_dir()]
    candidates = sorted(set(p.resolve() for p in candidates))
    if len(candidates) != 1:
        raise RuntimeError(
            f'Cần đúng một thư mục cifar-100-python, tìm thấy {len(candidates)}: {candidates}. '
            'Hãy đặt CIFAR_INPUT_HINT.'
        )
    return candidates[0]

cifar_source = discover_cifar_directory()
dataset_root = PROJECT_DIR / 'data' / 'datasets'
dataset_root.mkdir(parents=True, exist_ok=True)
cifar_target = dataset_root / 'cifar-100-python'
if cifar_target.is_symlink() or cifar_target.is_file():
    cifar_target.unlink()
elif cifar_target.exists():
    shutil.rmtree(cifar_target)
cifar_target.symlink_to(cifar_source, target_is_directory=True)

from torchvision.datasets import CIFAR100
train_set = CIFAR100(root=str(dataset_root), train=True, download=False)
test_set = CIFAR100(root=str(dataset_root), train=False, download=False)
print('CIFAR source:', cifar_source)
print('Train/Test  :', len(train_set), len(test_set))

## 3. Kiểm tra GPU và cài dependency

Không cài lại Torch/Torchvision của Kaggle. Nếu cell pip không truy cập được Internet, hãy bật Internet trong Settings hoặc gắn một wheelhouse riêng.

In [ ]:
subprocess.run(['nvidia-smi'], check=False)
import torch, torchvision
print('Python     :', sys.version.split()[0])
print('PyTorch    :', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA       :', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Vào Settings → Accelerator và bật GPU trước khi train.')
print('GPU        :', torch.cuda.get_device_name(0))

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed',
     '-r', 'requirements-colab.txt', '-r', 'requirements-qgors.txt'],
    check=True,
)
import timm, pennylane as qml
print('timm     :', timm.__version__)
print('PennyLane:', qml.__version__)
subprocess.run(
    [sys.executable, '-m', 'py_compile', 'main.py', 'trainer.py', 'models/base.py',
     'models/RSIAT_adapter.py', 'models/qgors.py'],
    check=True,
)
print('Compile OK')

## 4. Khôi phục checkpoint của phiên trước (tuỳ chọn)

Sau mỗi phiên, dùng **Save Version** để lưu `/kaggle/working/QGORS_runs`. Ở phiên tiếp theo, gắn output đó làm Input và đặt `PREVIOUS_RUN_INPUT_HINT` tới thư mục `QGORS_runs`.

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if PREVIOUS_RUN_INPUT_HINT:
    previous = Path(PREVIOUS_RUN_INPUT_HINT)
    if not previous.is_dir():
        raise FileNotFoundError(previous)
    shutil.copytree(previous, OUTPUT_ROOT, dirs_exist_ok=True)
    print('Đã khôi phục output từ:', previous)
else:
    print('Không khôi phục checkpoint cũ; đây là run mới hoặc smoke test.')

## 5. Sinh config dành cho Kaggle

Full config giữ nguyên epoch và hyperparameter nghiên cứu. `max_tasks_per_run` chỉ chia run thành nhiều phiên; checkpoint task-boundary đảm bảo phiên sau tiếp tục đúng task kế tiếp.

In [ ]:
def make_kaggle_config(source, destination, *, resume, max_tasks):
    config = json.loads(Path(source).read_text(encoding='utf-8'))
    config.update({
        'seed': [SEED],
        'resume': resume,
        'output_root': str(OUTPUT_ROOT),
        'device': ['0'],
        'num_workers': 4,
        'stats_num_workers': 2,
        'pin_memory': True,
        'persistent_workers': True,
        'max_tasks_per_run': max_tasks,
    })
    Path(destination).write_text(json.dumps(config, indent=2), encoding='utf-8')
    return config

smoke_config = make_kaggle_config(
    'exps/adapter_cifar224_qgors_smoke.json', '/kaggle/working/qgors_smoke.json',
    resume=False, max_tasks=2,
)
full_source = {
    'qgors': 'exps/adapter_cifar224_qgors.json',
    'classical': 'exps/adapter_cifar224_classical_gate.json',
}.get(FULL_VARIANT)
if full_source is None:
    raise ValueError('FULL_VARIANT phải là qgors hoặc classical')
full_config = make_kaggle_config(
    full_source, '/kaggle/working/qgors_full.json',
    resume=True, max_tasks=TASKS_PER_SESSION,
)
print('Smoke config:', json.dumps(smoke_config, indent=2))
print('Full config :', json.dumps(full_config, indent=2))

## 6. Micro-benchmark projector

Đây chỉ là overhead của projector, không phải ETA toàn bộ ViT. Nó giúp phát hiện sớm simulator quá chậm trên CPU được Kaggle cấp.

In [ ]:
from models.qgors import build_projector

bench_args = dict(full_config)
bench_args['projector_type'] = 'quantum'
projector = build_projector(bench_args, input_dims=768)
x = torch.randn(64, 768)
for _ in range(2):
    projector.zero_grad(set_to_none=True)
    projector(x).mean().backward()
start = time.perf_counter()
repeats = 10
for _ in range(repeats):
    projector.zero_grad(set_to_none=True)
    projector(x).mean().backward()
seconds = (time.perf_counter() - start) / repeats
print(f'Quantum projector batch=64: {seconds:.4f} giây/forward+backward')
del projector, x

## 7. Chạy smoke test hai task

Smoke dùng 1 epoch/stage và không được dùng làm kết quả báo cáo. Nếu smoke lỗi, không bật full run.

In [ ]:
if RUN_SMOKE:
    started = time.perf_counter()
    subprocess.run(
        [sys.executable, 'main.py', '--config', '/kaggle/working/qgors_smoke.json'],
        check=True,
    )
    print(f'Smoke hoàn tất sau {(time.perf_counter() - started) / 60:.1f} phút')
else:
    print('RUN_SMOKE=False — bỏ qua smoke test.')

## 8. Chạy một chặng full experiment

Đổi `RUN_FULL = True` ở cell Settings sau khi smoke thành công. Mỗi lần chạy xử lý tối đa `TASKS_PER_SESSION` task rồi lưu checkpoint. Không đổi prefix, seed hoặc projector khi resume.

In [ ]:
if RUN_FULL:
    started = time.perf_counter()
    subprocess.run(
        [sys.executable, 'main.py', '--config', '/kaggle/working/qgors_full.json'],
        check=True,
    )
    print(f'Chặng full hoàn tất sau {(time.perf_counter() - started) / 3600:.2f} giờ')
else:
    print('RUN_FULL=False — chưa chạy full experiment.')

## 9. Kiểm tra và đóng gói output

File ZIP bên dưới chứa log và checkpoint. Dữ liệu gốc trong `/kaggle/input` không được đóng gói lại.

In [ ]:
print('Checkpoint gần nhất:')
checkpoints = sorted(OUTPUT_ROOT.rglob('task_*.pkl'), key=lambda p: p.stat().st_mtime)
for path in checkpoints[-10:]:
    print(' -', path.relative_to(OUTPUT_ROOT))
print('Log gần nhất:')
logs = sorted(OUTPUT_ROOT.rglob('*.log'), key=lambda p: p.stat().st_mtime)
for path in logs[-10:]:
    print(' -', path.relative_to(OUTPUT_ROOT))

archive_base = Path('/kaggle/working/qgors_artifacts')
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_ROOT)
print('Artifact:', archive_path)
print('Nhớ Save Version để checkpoint trong /kaggle/working được lưu lại.')